[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yryo1005/OpenCampus_Demo/blob/main/OC_FaceLandmark.ipynb)


# 顔ランドマーク検出デモ（Face Landmark Detection）

カメラや写真から顔を検出し，目・鼻・口・輪郭などの**特徴点（ランドマーク）**を可視化するデモです．  
Google の MediaPipe Face Landmarker を使い，1 顔あたり最大 **478 点**（虹彩含む）を検出します．

**実行環境**: Google Colab（ランタイム → GPU: T4 推奨．CPU でも動作します）

## セルの進め方
1. **設定**（Webカメラの左右反転・描画オプション）
2. **ライブラリのインストール**
3. **ライブラリの読み込み・モデル準備・サンプル画像のダウンロード**
4. **Gradio の起動**

> API キーは **不要** です（推論はすべて Colab 内で完結）．  
> Hugging Face のユーザー認証も **不要** です．  
> インストール直後にエラーが出る場合は，**ランタイム → セッションを再起動**してから設定セルとセル3以降を再実行してください．


## 0. 設定

- カメラ映像が左右反転して見える場合は，次のセルの `MIRROR_WEBCAM` を切り替えてください（`True` = ミラー，`False` = 反転なし）．
- メッシュ／輪郭／虹彩の描画はオン／オフできます．
- 変更後は **Gradio 起動セル**を再実行してください．


In [ ]:
# Webカメラの左右反転（ミラー表示）
# True  : 左右反転する（Gradio のデフォルトに近い自撮り表示）
# False : 左右反転しない
MIRROR_WEBCAM = True

# 同時に検出する顔の最大数
MAX_NUM_FACES = 3

# 可視化オプション
DRAW_TESSELATION = True   # 顔全体のメッシュ（三角分割）
DRAW_CONTOURS = True      # 目・眉・唇・輪郭など
DRAW_IRISES = True        # 虹彩
DRAW_LANDMARK_DOTS = False  # 全点をドット表示（重い／見づらい場合は False）

# MediaPipe Face Landmarker モデル
MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "face_landmarker/face_landmarker/float16/1/face_landmarker.task"
)
MODEL_PATH = "models/face_landmarker.task"

print(f"MIRROR_WEBCAM = {MIRROR_WEBCAM}")
print(f"MAX_NUM_FACES = {MAX_NUM_FACES}")
print(f"DRAW_TESSELATION = {DRAW_TESSELATION}")
print(f"DRAW_CONTOURS = {DRAW_CONTOURS}")
print(f"DRAW_IRISES = {DRAW_IRISES}")
print(f"DRAW_LANDMARK_DOTS = {DRAW_LANDMARK_DOTS}")
print(f"MODEL_PATH = {MODEL_PATH}")


## 1. ライブラリのインストール


In [ ]:
# MediaPipe Face Landmarker を利用（Colab 標準の gradio / opencv / Pillow はそのまま）
!pip install -q -U "mediapipe>=0.10.14"


## 2. ライブラリの読み込み，変数のインスタンス化

サンプル顔写真（日本人・アジア系を含む）と Face Landmarker モデルをインターネットからダウンロードし，検出器を準備します．  
初回はモデルのダウンロードに数十秒かかることがあります．


In [ ]:
from __future__ import annotations

import urllib.request
from pathlib import Path

import cv2
import gradio as gr
import mediapipe as mp
import numpy as np
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
from PIL import Image, ImageDraw, ImageFont
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 定数・サンプル画像 URL
# ------------------------------------------------------------
SAMPLE_DIR = Path("samples_face_landmark")
FONT_DIR = Path("fonts")
FONT_PATH = FONT_DIR / "NotoSansJP-VF.ttf"
FONT_URL = (
    "https://raw.githubusercontent.com/googlefonts/noto-cjk/main/"
    "Sans/Variable/TTF/Subset/NotoSansJP-VF.ttf"
)

# 公開画像（Wikimedia / Pexels）．日本人・アジア系を含む．
SAMPLE_IMAGE_SOURCES: list[tuple[str, str, str]] = [
    (
        "happy_japanese_woman.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/b/b0/Smiling_Japanese_Woman.jpg",
        "日本人・笑顔",
    ),
    (
        "happy_japanese_smile.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/d/d1/Smiling_Ai_Hongo_%282024%2902.jpg",
        "日本人・スマイル",
    ),
    (
        "neutral_japanese.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/9/90/Geisha_face_%285025641801%29.jpg",
        "日本人・ポートレート",
    ),
    (
        "asian_portrait.jpg",
        "https://images.pexels.com/photos/1239291/pexels-photo-1239291.jpeg?auto=compress&cs=tinysrgb&w=640",
        "アジア系ポートレート",
    ),
    (
        "serious.jpg",
        "https://images.pexels.com/photos/2379004/pexels-photo-2379004.jpeg?auto=compress&cs=tinysrgb&w=640",
        "正面ポートレート",
    ),
]

USER_AGENT = (
    "Mozilla/5.0 (compatible; OpenCampusDemo/1.0; "
    "+https://github.com/yryo1005/OpenCampus_Demo)"
)

# 描画色（BGR ではなく RGB 画像に描くため RGB）
COLOR_MESH = (80, 200, 255)
COLOR_CONTOUR = (0, 220, 120)
COLOR_IRIS = (255, 180, 40)
COLOR_DOT = (255, 80, 80)


def download_bytes(url: str, save_path: Path, timeout: int = 120) -> Path:
    """URL からバイナリをダウンロードして保存する（既存ならスキップ）．

    Args:
        url (str): ダウンロード元 URL
        save_path (Path): 保存先パス
        timeout (int): タイムアウト秒

    Returns:
        Path: 保存したファイルのパス
    """
    if save_path.exists() and save_path.stat().st_size > 0:
        return save_path
    save_path.parent.mkdir(parents=True, exist_ok=True)
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    with urllib.request.urlopen(req, timeout=timeout) as response:
        save_path.write_bytes(response.read())
    return save_path


def download_image(url: str, save_path: Path, max_side: int = 1280) -> Path:
    """URL から画像をダウンロードし，必要なら長辺を縮小して保存する．

    Args:
        url (str): ダウンロード元 URL
        save_path (Path): 保存先パス
        max_side (int): 長辺の上限ピクセル（既定 1280）

    Returns:
        Path: 保存したファイルのパス
    """
    if save_path.exists() and save_path.stat().st_size > 0:
        return save_path
    save_path.parent.mkdir(parents=True, exist_ok=True)
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    with urllib.request.urlopen(req, timeout=60) as response:
        raw = response.read()
    arr = np.frombuffer(raw, dtype=np.uint8)
    bgr = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    if bgr is None:
        raise RuntimeError(f"画像のデコードに失敗しました: {url}")
    h, w = bgr.shape[:2]
    long_side = max(h, w)
    if long_side > max_side:
        scale = max_side / float(long_side)
        bgr = cv2.resize(
            bgr,
            (int(w * scale), int(h * scale)),
            interpolation=cv2.INTER_AREA,
        )
    ok, encoded = cv2.imencode(".jpg", bgr, [int(cv2.IMWRITE_JPEG_QUALITY), 90])
    if not ok:
        raise RuntimeError(f"画像のエンコードに失敗しました: {save_path}")
    save_path.write_bytes(encoded.tobytes())
    return save_path


def download_font(url: str, save_path: Path) -> Path:
    """日本語表示用フォントをダウンロードする（既存ならスキップ）．

    Args:
        url (str): フォントの URL
        save_path (Path): 保存先パス

    Returns:
        Path: 保存したフォントのパス
    """
    return download_bytes(url, save_path, timeout=120)


def prepare_sample_images(
    sources: list[tuple[str, str, str]],
    sample_dir: Path,
) -> list[tuple[str, Path]]:
    """サンプル顔写真をダウンロードし，ラベルとパスの一覧を返す．

    Args:
        sources (list[tuple[str, str, str]]): (ファイル名, URL, 表示ラベル) のリスト
        sample_dir (Path): 保存先ディレクトリ

    Returns:
        list[tuple[str, Path]]: (表示ラベル, ローカルパス) のリスト
    """
    prepared: list[tuple[str, Path]] = []
    for filename, url, label in tqdm(sources, desc="サンプル画像DL", leave=False):
        path = download_image(url, sample_dir / filename)
        print(f"  {label}: {path} ({path.stat().st_size} bytes)")
        prepared.append((label, path))
    return prepared


def load_face_landmarker(model_path: Path, num_faces: int) -> vision.FaceLandmarker:
    """MediaPipe Face Landmarker を構築する．

    Args:
        model_path (Path): .task モデルファイルのパス
        num_faces (int): 同時検出する顔の最大数

    Returns:
        vision.FaceLandmarker: 顔ランドマーク検出器
    """
    options = vision.FaceLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=str(model_path)),
        running_mode=vision.RunningMode.IMAGE,
        num_faces=num_faces,
        min_face_detection_confidence=0.5,
        min_face_presence_confidence=0.5,
        min_tracking_confidence=0.5,
        output_face_blendshapes=False,
        output_facial_transformation_matrixes=False,
    )
    return vision.FaceLandmarker.create_from_options(options)


def to_rgb_uint8(image) -> np.ndarray | None:
    """Gradio / PIL / ndarray 入力を RGB uint8 (H, W, 3) に揃える．

    Args:
        image: Gradio Image の入力（None / PIL.Image / np.ndarray）

    Returns:
        np.ndarray | None: RGB 画像．入力が無い場合は None
    """
    if image is None:
        return None
    if isinstance(image, Image.Image):
        return np.asarray(image.convert("RGB"))
    arr = np.asarray(image)
    if arr.ndim == 2:
        return cv2.cvtColor(arr.astype(np.uint8), cv2.COLOR_GRAY2RGB)
    if arr.shape[2] == 4:
        return arr[:, :, :3].astype(np.uint8)
    return arr.astype(np.uint8)


def landmark_to_pixel(
    landmark,
    width: int,
    height: int,
) -> tuple[int, int]:
    """正規化ランドマーク座標をピクセル座標へ変換する．

    Args:
        landmark: x, y 属性を持つランドマーク
        width (int): 画像幅
        height (int): 画像高さ

    Returns:
        tuple[int, int]: (x_px, y_px)
    """
    x = int(np.clip(landmark.x * width, 0, width - 1))
    y = int(np.clip(landmark.y * height, 0, height - 1))
    return x, y


def draw_connections(
    rgb: np.ndarray,
    landmarks: list,
    connections: list,
    color: tuple[int, int, int],
    thickness: int = 1,
) -> None:
    """ランドマーク間の接続線を RGB 画像へ描画する（インプレース）．

    Args:
        rgb (np.ndarray): RGB 画像，形状 (H, W, 3)
        landmarks (list): 1 顔分のランドマーク列
        connections (list): Connection(start, end) のリスト
        color (tuple[int, int, int]): RGB 色
        thickness (int): 線の太さ
    """
    h, w = rgb.shape[:2]
    for conn in connections:
        x1, y1 = landmark_to_pixel(landmarks[conn.start], w, h)
        x2, y2 = landmark_to_pixel(landmarks[conn.end], w, h)
        cv2.line(rgb, (x1, y1), (x2, y2), color, thickness, lineType=cv2.LINE_AA)


def draw_landmark_dots(
    rgb: np.ndarray,
    landmarks: list,
    color: tuple[int, int, int],
    radius: int = 1,
) -> None:
    """全ランドマークをドットで描画する（インプレース）．

    Args:
        rgb (np.ndarray): RGB 画像，形状 (H, W, 3)
        landmarks (list): 1 顔分のランドマーク列
        color (tuple[int, int, int]): RGB 色
        radius (int): 円の半径
    """
    h, w = rgb.shape[:2]
    for lm in landmarks:
        x, y = landmark_to_pixel(lm, w, h)
        cv2.circle(rgb, (x, y), radius, color, -1, lineType=cv2.LINE_AA)


def draw_face_label(rgb: np.ndarray, text: str, org: tuple[int, int]) -> np.ndarray:
    """日本語ラベルを描画した画像を返す．

    Args:
        rgb (np.ndarray): RGB 画像，形状 (H, W, 3)
        text (str): 表示文字列
        org (tuple[int, int]): 左上付近の描画位置 (x, y)

    Returns:
        np.ndarray: 描画後の RGB 画像，形状 (H, W, 3)
    """
    pil = Image.fromarray(rgb)
    draw = ImageDraw.Draw(pil)
    try:
        font = ImageFont.truetype(str(FONT_PATH), 26)
    except OSError:
        font = ImageFont.load_default()
    draw.text(org, text, fill=(20, 180, 90), font=font)
    return np.asarray(pil)


def format_detection_summary(face_landmarks_list: list) -> str:
    """検出結果の要約テキストを作る．

    Args:
        face_landmarks_list (list): 各顔のランドマーク列のリスト

    Returns:
        str: 高校生向けの説明テキスト
    """
    n_faces = len(face_landmarks_list)
    if n_faces == 0:
        return (
            "顔を検出できませんでした．\n"
            "顔が正面・明るく・大きく写るようにして，もう一度お試しください．"
        )

    lines = [f"【検出結果】顔の数: {n_faces}"]
    for i, landmarks in enumerate(face_landmarks_list, start=1):
        n_pts = len(landmarks)
        # 代表点: 鼻先(1), 左目外端付近, 右目外端付近 はモデル依存のため座標統計を出す
        xs = [lm.x for lm in landmarks]
        ys = [lm.y for lm in landmarks]
        zs = [lm.z for lm in landmarks]
        lines.append(
            f"\n顔 {i}: ランドマーク {n_pts} 点"
            f"（x平均 {np.mean(xs):.3f}, y平均 {np.mean(ys):.3f}, "
            f"z平均 {np.mean(zs):.4f}）"
        )

    lines.append(
        "\nヒント: ランドマークは AR フィルタやアバター，視線推定などに使われます．"
        "点のつながり（メッシュ）が顔の立体構造を近似しています．"
    )
    return "\n".join(lines)


def detect_and_draw(image, mirror: bool = False) -> tuple[np.ndarray | None, str]:
    """顔写真からランドマークを検出し可視化する（Gradio コールバック）．

    Args:
        image: Gradio Image 入力（カメラ／アップロード／サンプル）
        mirror (bool): True のとき左右反転してから処理する

    Returns:
        tuple[np.ndarray | None, str]: (可視化画像, 結果テキスト)
    """
    rgb = to_rgb_uint8(image)
    if rgb is None:
        return None, (
            "画像がありません．カメラ撮影・アップロード・サンプルのいずれかを選んでください．"
        )

    if mirror:
        rgb = np.ascontiguousarray(rgb[:, ::-1, :])
    else:
        rgb = np.ascontiguousarray(rgb)

    with tqdm(total=2, desc="顔ランドマーク", leave=False) as pbar:
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = face_landmarker.detect(mp_image)
        pbar.update(1)

        vis = rgb.copy()
        faces = result.face_landmarks
        Conn = vision.FaceLandmarksConnections

        for face_idx, landmarks in enumerate(faces):
            if DRAW_TESSELATION:
                draw_connections(
                    vis, landmarks, Conn.FACE_LANDMARKS_TESSELATION, COLOR_MESH, 1
                )
            if DRAW_CONTOURS:
                draw_connections(
                    vis, landmarks, Conn.FACE_LANDMARKS_CONTOURS, COLOR_CONTOUR, 2
                )
            if DRAW_IRISES:
                draw_connections(
                    vis, landmarks, Conn.FACE_LANDMARKS_LEFT_IRIS, COLOR_IRIS, 2
                )
                draw_connections(
                    vis, landmarks, Conn.FACE_LANDMARKS_RIGHT_IRIS, COLOR_IRIS, 2
                )
            if DRAW_LANDMARK_DOTS:
                draw_landmark_dots(vis, landmarks, COLOR_DOT, radius=1)

            # 顔ラベル位置: 最上点付近
            h, w = vis.shape[:2]
            top = min(landmarks, key=lambda lm: lm.y)
            tx, ty = landmark_to_pixel(top, w, h)
            vis = draw_face_label(
                vis, f"顔{face_idx + 1} ({len(landmarks)}点)", (tx, max(0, ty - 32))
            )

        pbar.update(1)

    text = format_detection_summary(faces)
    return vis, text


def build_demo(sample_items: list[tuple[str, Path]]) -> gr.Blocks:
    """Gradio UI を構築する．

    Args:
        sample_items (list[tuple[str, Path]]): (表示ラベル, 画像パス)

    Returns:
        gr.Blocks: Gradio デモ
    """
    example_paths = [str(path) for _, path in sample_items]

    with gr.Blocks(title="顔ランドマーク検出デモ") as demo:
        gr.Markdown(
            """
            # 顔ランドマーク検出デモ
            顔写真から **目・鼻・口・輪郭**などの特徴点を検出し，メッシュとして重ねて表示します．  
            **カメラ**で撮影するか，下の**サンプル画像**をクリックして試せます．
            """
        )
        with gr.Row():
            with gr.Column():
                image_in = gr.Image(
                    label="顔写真（カメラ / アップロード）",
                    type="numpy",
                    sources=["webcam", "upload"],
                    webcam_options=gr.WebcamOptions(mirror=MIRROR_WEBCAM),
                )
                mirror_flag = gr.Checkbox(
                    label="入力画像を左右反転して検出する",
                    value=False,
                    info=(
                        "アップロード画像の向きが逆のときだけオンにしてください"
                        "（カメラは上のミラー設定を利用）"
                    ),
                )
                run_btn = gr.Button("ランドマークを検出", variant="primary")
            with gr.Column():
                image_out = gr.Image(label="検出結果（メッシュ）", type="numpy")
                text_out = gr.Textbox(label="検出サマリー", lines=10)

        gr.Examples(
            examples=example_paths,
            inputs=[image_in],
            label="サンプル画像（クリックで入力）",
            examples_per_page=8,
        )

        run_btn.click(
            fn=detect_and_draw,
            inputs=[image_in, mirror_flag],
            outputs=[image_out, text_out],
        )
        image_in.change(
            fn=detect_and_draw,
            inputs=[image_in, mirror_flag],
            outputs=[image_out, text_out],
        )
    return demo


# ------------------------------------------------------------
# 初期化
# ------------------------------------------------------------
model_path = Path(MODEL_PATH)
print("モデルをダウンロード中...")
download_bytes(MODEL_URL, model_path)
print(f"モデル: {model_path} ({model_path.stat().st_size} bytes)")

download_font(FONT_URL, FONT_PATH)
print(f"フォント: {FONT_PATH} ({FONT_PATH.stat().st_size} bytes)")

sample_items = prepare_sample_images(SAMPLE_IMAGE_SOURCES, SAMPLE_DIR)
face_landmarker = load_face_landmarker(model_path, num_faces=MAX_NUM_FACES)
print("初期化完了．次のセルで Gradio を起動してください．")


## 3. Gradio の実行

UI が起動したら，サンプル画像をクリックするか，カメラで顔を撮影して「ランドマークを検出」を押してください．


In [ ]:
demo = build_demo(sample_items)
demo.launch(share=True, debug=False)
